In [ ]:
import pandas as pd
import numpy as np
from plotnine import *

from vpop_calibration.api import *

%load_ext autoreload
%autoreload 2

In [ ]:
model = SimworkModelBinding(
    path_to_model="cm.json",
    inputs=[
        "ka",
        "V",
        "Vm",
        "Km",
        "Beta",
        "Dose"
    ],
    outputs=["C","cumulative_hazard"],
    path_to_solving_options="sv.json",
)

protocol_design = pd.DataFrame({"protocol_arm": ["arm-A", "arm-B"], "Dose": [50, 100]})
struct_model = StructuralSimwork(protocol_design = protocol_design, model= model)

In [ ]:
param_distrib = {
    "pdu": {
        "ka": {"prior": 0.5, "prior_omega": 0.3},
        "V": {"prior": 70, "prior_omega": 0.2},
        "Vm": {"prior": 6, "prior_omega": 0.1},
        "Km": {"prior": 0.2, "prior_omega": 0.2},
        "Beta": {"prior": 0.5, "prior_omega": 0.1},
    },
    "error_model":{
        "C": {"error_type": "combined", "sigma_add": 0.05, "sigma_prop":0.05},
        "cumulative_hazard": {"error_type": "additive", "sigma": 0},
    },
}
nb_patients = 50
time = [0.5,4.0,8.0,12.0,16.0,20.0,24.0,36.0,60.0,84.0,108.0,132.0,144.0,148.0,152.0,156.0,160.0,164.0,168.0,172.0] 
time = [t * 3600 for t in time]

In [ ]:
synthetic_df = generate_synthetic_data(
    struct_model=struct_model,
    param_distrib=param_distrib,
    nb_patients=nb_patients,
    time=time,
)

In [ ]:
synthetic_df

In [ ]:
obs_df = synthetic_df.loc[synthetic_df["output_name"] != "cumulative_hazard"].copy()

In [ ]:
rng = np.random.default_rng(0)

lam = (
    synthetic_df.loc[synthetic_df["output_name"] == "lambda", ["id", "time", "value"]]
    .drop_duplicates(subset=["id", "time"])
    .sort_values(["id", "time"])
    .reset_index(drop=True)
)

surv_df = lam.rename(columns={"time": "event_time"}).drop(columns="value") 
cumh = (
    synthetic_df.loc[synthetic_df["output_name"] == "cumulative_hazard", ["id", "time", "value"]]
    .sort_values(["id", "time"])
)
dLambda = cumh.groupby("id")["value"].diff().fillna(cumh["value"]).to_numpy()
draw = rng.binomial(1, 1 - np.exp(-dLambda)).astype(bool)
cumh = cumh.assign(event=draw)

rows = []
for pid, g in cumh.groupby("id"):
    hits = g.loc[g["event"]]
    if len(hits):
        rows.append((pid, hits["time"].iloc[0], True))
    else:
        rows.append((pid, g["time"].iloc[-1], False))
surv_df = pd.DataFrame(rows, columns=["id", "event_time", "event_status"])
surv_df["hazard_name"] = "hazard"
surv_df

In [ ]:
training_df = obs_df.merge(surv_df, on="id")

In [ ]:
training_df.to_csv("data.csv", index=False)